# Day 4 — Fake Data Test (Three Observables)

Goal: simulate the SIR narrative-contagion model with known parameters, generate synthetic Google Trends, Article-count, and Tone series, then verify that the UKF + maximum-likelihood estimator recovers the true parameters and separately identifies process noise **Q** from observation noise **R**.

State: $x_t = [S_t, I_t]^T$.  Observation: $y_t = [GT_t, A_t, T_t]^T$.

$$
\begin{aligned}
S_{t+1} &= S_t - \beta S_t I_t \Delta t + w_S \\
I_{t+1} &= I_t + \beta S_t I_t \Delta t - \gamma I_t \Delta t + w_I \\
GT_t &= 1 \cdot I_t + \varepsilon_1 \\
A_t  &= c_2 \cdot I_t + \varepsilon_2 \\
T_t  &= c_3 \cdot I_t + \varepsilon_3, \quad c_3 < 0
\end{aligned}
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from filterpy.kalman import MerweScaledSigmaPoints, UnscentedKalmanFilter as UKF

rng = np.random.default_rng(20260430)

## 1. Transition and observation functions

In [ ]:
DT = 1.0  # one time step

def fx(x, dt, beta, gamma):
    """Euler-discretized SIR transition on the 2D state [S, I]."""
    S, I = x
    new_S = S - beta * S * I * dt
    new_I = I + beta * S * I * dt - gamma * I * dt
    return np.array([new_S, new_I])

def hx_full(x, c2, c3):
    """Observation function when GT, Articles and Tone are all available."""
    I = x[1]
    return np.array([1.0 * I, c2 * I, c3 * I])

def hx_gt_only(x, c2, c3):
    """Partial observation when only Google Trends is available."""
    I = x[1]
    return np.array([1.0 * I])

## 2. Simulate the truth

Pick true parameters in line with Phase 6 of the spec ($\beta = 0.3$, $\gamma = 0.1$). $c_2$ scales Google-Trends units to article counts; $c_3$ is negative because more engagement drives tone more negative. Process noise is small to keep $S, I \in [0,1]$.

In [ ]:
true_params = dict(
    beta=0.30,
    gamma=0.10,
    c2=80.0,        # ~80 articles per GT unit
    c3=-0.05,       # tone shifts ~ -5 per GT unit (scaled)
    sigma2_S=1e-6,
    sigma2_I=1e-6,
    sigma2_e1=4.0,      # GT noise variance
    sigma2_e2=200.0**2, # article-count noise variance
    sigma2_e3=0.5**2,   # tone noise variance
    I0=0.001,
)
T = 200  # time steps

def simulate(params, T, seed=0):
    r = np.random.default_rng(seed)
    S = np.zeros(T); I = np.zeros(T)
    S[0] = 1.0 - params['I0']
    I[0] = params['I0']
    Q = np.diag([params['sigma2_S'], params['sigma2_I']])
    for t in range(1, T):
        x = fx([S[t-1], I[t-1]], DT, params['beta'], params['gamma'])
        w = r.multivariate_normal([0, 0], Q)
        S[t] = np.clip(x[0] + w[0], 1e-9, 1.0)
        I[t] = np.clip(x[1] + w[1], 1e-9, 1.0)
    GT  = 1.0 * I + r.normal(0, np.sqrt(params['sigma2_e1']), T)
    A   = params['c2'] * I + r.normal(0, np.sqrt(params['sigma2_e2']), T)
    Tn  = params['c3'] * I + r.normal(0, np.sqrt(params['sigma2_e3']), T)
    return S, I, np.column_stack([GT, A, Tn])

S_true, I_true, Y = simulate(true_params, T, seed=1)

fig, ax = plt.subplots(2, 2, figsize=(11, 6))
ax[0,0].plot(I_true, label='I(t) true'); ax[0,0].set_title('Latent infected I(t)'); ax[0,0].legend()
ax[0,1].plot(Y[:,0]); ax[0,1].set_title('Synthetic Google Trends')
ax[1,0].plot(Y[:,1]); ax[1,0].set_title('Synthetic Article counts')
ax[1,1].plot(Y[:,2]); ax[1,1].set_title('Synthetic Tone (negative)')
plt.tight_layout(); plt.show()

## 3. Inject a missing-data pattern

Drop 15 random observations of Articles **and** Tone (GDELT outage), keeping GT always available. The mask tells the filter which dimensions of $y_t$ are real on each step.

In [ ]:
mask = np.ones_like(Y, dtype=bool)
miss_idx = rng.choice(np.arange(5, T), size=15, replace=False)
mask[miss_idx, 1] = False
mask[miss_idx, 2] = False
Y_masked = Y.copy()
Y_masked[~mask] = np.nan
print('Missing GDELT days:', np.sort(miss_idx))

## 4. UKF + log-likelihood

The UKF predicts forward via `fx`. On each step we slice the observation matrix and `R` to whatever rows are available (3D update if GDELT present, 1D update if only GT).

In [ ]:
def make_ukf(theta):
    """Build a fresh UKF parameterized by theta."""
    beta, gamma, c2, c3, s2S, s2I, s2e1, s2e2, s2e3, I0 = theta
    sp = MerweScaledSigmaPoints(n=2, alpha=1e-3, beta=2.0, kappa=0.0)
    f = UKF(dim_x=2, dim_z=3, dt=DT,
            fx=lambda x, dt: fx(x, dt, beta, gamma),
            hx=lambda x: hx_full(x, c2, c3),
            points=sp)
    f.x = np.array([1.0 - I0, I0])
    f.P = np.diag([1e-4, 1e-4])
    f.Q = np.diag([s2S, s2I])
    f.R = np.diag([s2e1, s2e2, s2e3])
    return f, (c2, c3)

def neg_log_lik(theta_unc, Y, mask):
    # Unconstrained -> constrained via softplus / sign tricks
    beta   = np.exp(theta_unc[0])
    gamma  = np.exp(theta_unc[1])
    c2     = np.exp(theta_unc[2])
    c3     = -np.exp(theta_unc[3])      # forced negative
    s2S    = np.exp(theta_unc[4])
    s2I    = np.exp(theta_unc[5])
    s2e1   = np.exp(theta_unc[6])
    s2e2   = np.exp(theta_unc[7])
    s2e3   = np.exp(theta_unc[8])
    I0     = 1.0 / (1.0 + np.exp(-theta_unc[9]))   # in (0,1)

    theta = (beta, gamma, c2, c3, s2S, s2I, s2e1, s2e2, s2e3, I0)
    f, (c2, c3) = make_ukf(theta)
    R_full = f.R.copy()

    ll = 0.0
    for t in range(Y.shape[0]):
        f.predict()
        avail = np.where(mask[t])[0]
        if len(avail) == 0:
            continue
        z = Y[t, avail]
        if len(avail) == 3:
            f.hx = lambda x: hx_full(x, c2, c3)
            f.R = R_full
            f.update(z)
        elif np.array_equal(avail, [0]):
            f.hx = lambda x: hx_gt_only(x, c2, c3)
            f.R = np.array([[R_full[0,0]]])
            f.dim_z = 1
            f.update(z)
            f.dim_z = 3   # restore for next full step
        else:
            # general partial slice (not used here but kept for completeness)
            H_idx = avail
            f.hx = lambda x, idx=H_idx: hx_full(x, c2, c3)[idx]
            f.R = R_full[np.ix_(H_idx, H_idx)]
            f.dim_z = len(H_idx)
            f.update(z)
            f.dim_z = 3
        # innovation contribution
        nu = f.y
        Pyy = f.S
        sign, logdet = np.linalg.slogdet(Pyy)
        ll += -0.5 * (logdet + nu @ np.linalg.solve(Pyy, nu) + len(nu) * np.log(2*np.pi))
    return -ll

## 5. Optimize

In [ ]:
# Initial guess deliberately offset from truth
theta0_unc = np.array([
    np.log(0.5),     # beta
    np.log(0.2),     # gamma
    np.log(50.0),    # c2
    np.log(0.1),     # |c3|
    np.log(1e-5),    # sigma2_S
    np.log(1e-5),    # sigma2_I
    np.log(2.0),     # sigma2_e1
    np.log(150.0**2),# sigma2_e2
    np.log(0.3**2),  # sigma2_e3
    np.log(0.005/(1-0.005)),  # I0 logit
])

res = minimize(neg_log_lik, theta0_unc, args=(Y_masked, mask),
               method='Nelder-Mead',
               options={'maxiter': 4000, 'xatol': 1e-5, 'fatol': 1e-5, 'disp': True})
print('Converged:', res.success, '  -logL =', res.fun)

## 6. Decode & compare

In [ ]:
u = res.x
est = dict(
    beta=np.exp(u[0]), gamma=np.exp(u[1]),
    c2=np.exp(u[2]),  c3=-np.exp(u[3]),
    sigma2_S=np.exp(u[4]), sigma2_I=np.exp(u[5]),
    sigma2_e1=np.exp(u[6]), sigma2_e2=np.exp(u[7]), sigma2_e3=np.exp(u[8]),
    I0=1/(1+np.exp(-u[9])),
)

print(f"{'param':<10}{'truth':>14}{'estimate':>14}{'rel.err %':>12}")
for k in ['beta','gamma','c2','c3','sigma2_S','sigma2_I',
         'sigma2_e1','sigma2_e2','sigma2_e3','I0']:
    t, e = true_params[k], est[k]
    err = 100 * (e - t) / (abs(t) + 1e-12)
    print(f"{k:<10}{t:>14.6g}{e:>14.6g}{err:>12.2f}")

## 7. Filtered I(t) vs truth, and Q-vs-R identifiability check

Re-run the UKF at the MLE so we can plot the filtered latent state against the truth, and verify that the filter is not collapsing process noise into observation noise (or vice versa).

In [ ]:
theta_mle = (est['beta'], est['gamma'], est['c2'], est['c3'],
             est['sigma2_S'], est['sigma2_I'],
             est['sigma2_e1'], est['sigma2_e2'], est['sigma2_e3'], est['I0'])
f, (c2, c3) = make_ukf(theta_mle)
R_full = f.R.copy()

I_filt = np.zeros(T); I_var = np.zeros(T)
for t in range(T):
    f.predict()
    avail = np.where(mask[t])[0]
    if len(avail) == 3:
        f.hx = lambda x: hx_full(x, c2, c3); f.R = R_full
        f.update(Y_masked[t])
    elif len(avail) == 1:
        f.hx = lambda x: hx_gt_only(x, c2, c3)
        f.R = np.array([[R_full[0,0]]]); f.dim_z = 1
        f.update(Y_masked[t, avail])
        f.dim_z = 3
    I_filt[t] = f.x[1]; I_var[t] = f.P[1,1]

fig, ax = plt.subplots(1, 1, figsize=(9,4))
ax.plot(I_true, 'k-', label='I(t) truth')
ax.plot(I_filt, 'C1--', label='I(t) UKF mean')
se = np.sqrt(np.maximum(I_var, 0))
ax.fill_between(np.arange(T), I_filt-2*se, I_filt+2*se, color='C1', alpha=0.2, label='\u00b12 SE')
for m in miss_idx: ax.axvline(m, color='red', alpha=0.15)
ax.set_title('Filtered I(t) vs truth (red lines = missing GDELT days)')
ax.legend(); plt.tight_layout(); plt.show()

print('\nQ vs R identifiability check (estimated / true):')
for k in ['sigma2_S','sigma2_I','sigma2_e1','sigma2_e2','sigma2_e3']:
    print(f'  {k}: {est[k]/true_params[k]:.3f}')

## 8. Pass criteria

- Relative error on $\beta, \gamma, c_2, c_3$ within ~10% — this is what tells us the structural parameters are recovered.
- Process-noise variances $\sigma^2_S, \sigma^2_I$ and observation-noise variances $\sigma^2_{\varepsilon\,k}$ stay in their own lanes (estimated/true ratios all close to 1, none collapsing to zero) — this confirms Q and R are separately identified, the question Phase 6 of the spec asks us to check.
- Filtered $I(t)$ tracks the truth and the credible band widens on the red (missing-GDELT) days, then re-tightens when GDELT returns.

If any of these fails: rerun with a different seed and a tighter optimizer (e.g. L-BFGS-B from the Nelder-Mead solution, or `differential_evolution` for a global search) before assuming a structural identifiability problem.